### Loading SmolLM2

This section loads the `SmolLM2-135M` causal language model from the Hugging Face Transformers library, allowing us to use a pre-trained model for further experimentation and training.

In [2]:
from transformers import AutoModelForCausalLM

hf_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")
print(hf_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

### Llama Architecture

This section defines the core components of a Llama-like model from scratch using PyTorch. It includes:

*   **RMSNorm**: An efficient normalization layer used instead of LayerNorm.
*   **RotaryEmbedding**: Implements Rotary Positional Embeddings (RoPE) to inject positional information into attention keys and queries.
*   **LlamaAttention**: The self-attention mechanism, incorporating RoPE and Grouped Query Attention (GQA) for efficiency.
*   **LlamaMLP**: The feed-forward network, using the SiLU activation function.
*   **LlamaDecoderLayer**: Combines LlamaAttention and LlamaMLP, along with RMSNorm, to form a single transformer block.
*   **LlamaModel**: The complete model, stacking multiple `LlamaDecoderLayer` instances, an initial embedding layer, and a final language model head.

#### **Essential** PyTorch libraries for building and training neural networks. torch is the main library, torch.nn provides modules for neural network layers, and torch.nn.functional offers functional interfaces for common operations.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

The `RMSNorm` layer is a normalization technique that scales the input based on the root mean square of its elements. Unlike LayerNorm, it doesn't subtract the mean, which can make it computationally more efficient. This implementation includes:

*   **`__init__(self, embedd_dim, eps=1e-05)`**: Initializes the layer. `embedd_dim` is the size of the embedding dimension, and `eps` is a small value added to the denominator for numerical stability.
    *   `self.weight = nn.Parameter(torch.ones(embedd_dim))`: Creates a learnable scaling parameter `weight` for each element in the `embedd_dim`. This allows the model to learn optimal scaling for each feature.
*   **`forward(self, x)`**: Defines the forward pass of the layer.
    *   `x = x / (torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True)) + self.eps)`: This is the core RMS normalization step. It calculates the mean of the squared values of `x` along the last dimension (`dim=-1`), takes the square root, and then divides `x` by this value (plus `self.eps` for stability). This effectively scales `x` by its RMS value.
    *   `x = self.weight * x`: The normalized `x` is then multiplied by the learnable `self.weight` parameter, allowing the model to introduce scaling factors.

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, embedd_dim, eps=1e-05):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(embedd_dim))
        self.eps = eps

    def forward(self, x):
        x = x / (torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True)) + self.eps)
        x = self.weight * x
        return x

#### **RotaryEmbedding**
The code in this cell implements **Rotary Positional Embeddings (RoPE)**, a technique used to incorporate positional information into self-attention mechanisms in transformer models.

##### `RotaryEmbedding` Class

This class is responsible for generating the cosine and sine rotation matrices based on input positions. It includes:

*   **`__init__(self, head_dim=64, rope_theta=100000)`**: The constructor initializes `head_dim` (the dimension of each attention head) and `rope_theta` (a hyperparameter controlling the frequency decay). It then precomputes `inv_freq`, which are inverse frequencies used to create the rotary sinusoids.
    *   `inv_freq = 1.0 / (rope_theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))`: This line calculates a series of inverse frequencies. These frequencies determine how quickly the rotation changes with position, and they are designed to be different for different dimensions within the head.
*   **`forward(self, x, position_ids)`**: This method takes the input tensor `x` (representing queries or keys) and `position_ids` (the absolute positions of tokens in the sequence) to generate the cosine and sine components for rotation.
    *   It expands `inv_freq` and `position_ids` to match dimensions, then computes `freqs` (element-wise products of inverse frequencies and position IDs). This `freqs` tensor essentially contains the arguments for the sine and cosine functions for each position and dimension.
    *   `emb = torch.cat((freqs, freqs), dim=-1)`: The `freqs` are duplicated and concatenated. This is because RoPE applies a 2D rotation to pairs of dimensions, so each frequency needs to be applied to two dimensions.
    *   `cos = emb.cos()` and `sin = emb.sin()`: These lines compute the cosine and sine of the `emb` tensor, which will be used to rotate the query and key vectors.

#####  `apply_rotary_pos_emb` Function

This function applies the rotary embeddings (cosine and sine) generated by `RotaryEmbedding` to the query (`q`) and key (`k`) tensors.

*   `cos = cos.unsqueeze(1)` and `sin = sin.unsqueeze(1)`: The cosine and sine tensors are expanded to match the dimensions of `q` and `k` for broadcasting.
*   `q_embed = (q * cos) + (rotate_half(q) * sin)` and `k_embed = (k * cos) + (rotate_half(k) * sin)`: This is the core RoPE application. It performs a 2D rotation. For each pair of dimensions in `q` and `k`, it combines the original values with the 'rotated half' values using the cosine and sine components. This effectively injects positional information.

#####  `rotate_half` Function

This is a helper function used by `apply_rotary_pos_emb` to rearrange a tensor for the RoPE rotation.

*   `x1 = x[..., :x.shape[-1]//2]` and `x2 = x[..., x.shape[-1]//2:]`: The input tensor `x` is split into two halves along its last dimension.
*   `return torch.cat((-x2, x1), dim=-1)`: The second half (`x2`) is negated and then concatenated with the first half (`x1`). This creates the 'rotated half' vector required for the 2D rotation formula.

In [5]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim=64, rope_theta=100000):
        super().__init__()
        self.head_dim = head_dim
        self.rope_theta = rope_theta
        # Precompute inverse frequencies
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x, position_ids):
        # x: (batch, num_heads, seq_len, head_dim)
        # position_ids: (batch, seq_len)
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1)
        position_ids_expanded = position_ids[:, None, :].float()

        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)  # Duplicate to match head_dim
        cos = emb.cos()
        sin = emb.sin()

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)

def apply_rotary_pos_emb(q, k, cos, sin):
    # Reshape cos/sin to match q/k shape
    cos = cos.unsqueeze(1)  # (batch, 1, seq_len, head_dim)
    sin = sin.unsqueeze(1)

    # Split and rotate
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

def rotate_half(x):
    # Rotate half the dimensions
    x1 = x[..., :x.shape[-1]//2]
    x2 = x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)


#### **LlamaAttention** class
This code defines the `LlamaAttention` class, which is the core self-attention mechanism for the Llama-like model. It handles how the model processes sequences by focusing on different parts of the input. Let's break down what each part does:

### `__init__` Method

*   It initializes the attention mechanism with `hidden_size`, `num_attention_heads`, `num_key_value_heads`, and `rope_theta`.
*   It sets up linear projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`) for generating queries, keys, values, and the final output. Notably, `k_proj` and `v_proj` are designed for Grouped Query Attention (GQA), where fewer key/value heads exist than query heads.
*   It initializes the `RotaryEmbedding` layer, which will be used to inject positional information.

### `forward` Method

1.  **Projecting to Q, K, V**: The input `x` (representing the sequence of tokens) is passed through the `q_proj`, `k_proj`, and `v_proj` linear layers to obtain the query (`q`), key (`k`), and value (`v`) representations.
2.  **Reshaping into Heads**: These `q`, `k`, and `v` tensors are then reshaped to separate the attention heads, preparing them for parallel processing in multi-head attention. The `transpose` operation reorders dimensions to get the correct shape for attention calculations.
3.  **Applying Rotary Positional Embeddings (RoPE)**: `position_ids` are generated, and then the `self.rope` module is used to compute cosine and sine components. These are then applied to the query (`q`) and key (`k`) tensors using `apply_rotary_pos_emb`, injecting positional information without using absolute position embeddings.
4.  **Repeat K/V Heads for GQA**: If `num_attention_heads` is greater than `num_key_value_heads` (i.e., using Grouped Query Attention), the key and value heads are repeated to match the number of query heads. This reduces computational cost while retaining most of the performance benefits of multi-head attention.
5.  **Compute Attention**: The `F.scaled_dot_product_attention` function calculates the attention weights and applies them to the value vectors. `is_causal=True` ensures that each token can only attend to previous tokens in the sequence, which is crucial for language modeling.
6.  **Reshape Back**: The `attn_output` is reshaped back to its original `(batch, seq_len, hidden_size)` format by transposing and concatenating the heads.
7.  **Output Projection**: Finally, the aggregated attention output is passed through the `o_proj` linear layer to produce the final output of the attention block.

In [6]:
class LlamaAttention(nn.Module):

    def __init__(self,hidden_size,num_attention_heads=9,num_key_value_heads=3, rope_theta=10000):
        super().__init__()
        assert hidden_size%num_attention_heads == 0
        assert hidden_size%num_key_value_heads == 0
        self.hidden_size = hidden_size
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.head_dim = self.hidden_size//self.num_attention_heads
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, self.num_key_value_heads*self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_size, self.num_key_value_heads*self.head_dim, bias=False)
        self.o_proj = nn.Linear(hidden_size,hidden_size, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, rope_theta)

    def forward(self, x):
        batch, seq_len, _ = x.shape

        # Project to Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Reshape into heads
        q = q.view(batch, seq_len, self.num_attention_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        # Generate position_ids and apply RoPE
        position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch, -1)
        cos, sin = self.rope(q, position_ids)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        # Repeat K/V heads for grouped query attention
        k = k.repeat_interleave(self.num_attention_heads // self.num_key_value_heads, dim=1)
        v = v.repeat_interleave(self.num_attention_heads // self.num_key_value_heads, dim=1)

        # Compute attention
        attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # Reshape back
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch, seq_len, -1)
        attn_output = self.o_proj(attn_output)

        return attn_output

#### **LlamaMLP**

This code defines the `LlamaMLP` class, which implements the feed-forward network (FFN) component within each Llama decoder layer. This specific architecture is a type of Gated Linear Unit (GLU) using the SiLU activation function, often referred to as a SwiGLU.

##### `__init__` Method

*   **`hidden_size`**: The dimension of the input and output features for the MLP.
*   **`intermediate_size`**: The hidden dimension within the MLP, which is typically larger than `hidden_size`.
*   **`self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)`**: This is the 'gate' projection. It transforms the input `x` to an `intermediate_size` dimension, and its output is then passed through a SiLU activation.
*   **`self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)`**: This is the 'up' projection. It also transforms the input `x` to an `intermediate_size` dimension.
*   **`self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)`**: This is the 'down' projection. It takes the element-wise product of the gated output and the `up_proj` output, and projects it back to the original `hidden_size`.
*   **`self.silu = nn.SiLU()`**: The SiLU (Sigmoid Linear Unit) activation function, which is applied to the output of `gate_proj`.

##### `forward` Method

1.  **`x_1 = self.silu(self.gate_proj(x))`**: The input `x` is passed through the `gate_proj` linear layer, and then the SiLU activation function is applied. This creates a gating mechanism.
2.  **`x_2 = self.up_proj(x)`**: The input `x` is also passed through the `up_proj` linear layer.
3.  **`return self.down_proj(x_1 * x_2)`**: The outputs `x_1` and `x_2` are multiplied element-wise. This gated product is then passed through the `down_proj` linear layer to produce the final output of the MLP. This gating mechanism allows the network to control the flow of information more effectively.

In [7]:
class LlamaMLP(nn.Module):

    def __init__(self,hidden_size=576,intermediate_size=1536):
        super().__init__()
        self.gate_proj =  nn.Linear(hidden_size,intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size,intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size,bias=False)
        self.silu = nn.SiLU()

    def forward(self,x):
        x_1 = self.silu(self.gate_proj(x))
        x_2 = self.up_proj(x)
        return self.down_proj(x_1*x_2)

#### **LlamaDecoderLayer**
This code defines the `LlamaDecoderLayer` class, which is a fundamental building block of the Llama model's transformer architecture. Each `LlamaDecoderLayer` processes its input `x` through two main sub-layers, each followed by a residual connection and RMS Normalization:

##### `__init__` Method

*   **`self.self_attn`**: Initializes an instance of `LlamaAttention`, which handles the self-attention mechanism, including RoPE and GQA.
*   **`self.mlp`**: Initializes an instance of `LlamaMLP`, which is the feed-forward network component.
*   **`self.input_layernorm`**: Initializes an `RMSNorm` layer applied before the self-attention block.
*   **`self.post_attention_layernorm`**: Initializes an `RMSNorm` layer applied before the MLP block.

##### `forward` Method

The `forward` method implements the sequential processing within a single decoder layer:

1.  **Attention Sub-layer**: The input `x` is first normalized by `self.input_layernorm`. The result is then passed through `self.self_attn`. The output of the attention mechanism is added back to the original `x` (residual connection): `x = x + self.self_attn(self.input_layernorm(x))`.

2.  **MLP Sub-layer**: The output from the attention sub-layer (which has already incorporated the residual connection and normalization) is then normalized by `self.post_attention_layernorm`. This normalized input is passed through `self.mlp`. The output of the MLP is added back to the current `x` (another residual connection): `x = x + self.mlp(self.post_attention_layernorm(x))`.

This structure ensures that information is effectively transformed and integrated across the model's layers while maintaining numerical stability through normalization and preventing vanishing gradients via residual connections.

In [8]:
class LlamaDecoderLayer(nn.Module):
    def __init__(self, hidden_size=576,intermediate_size=1536,num_attention_heads=9,num_key_value_heads=3, rope_theta=10000):
        super().__init__()
        self.self_attn = LlamaAttention(hidden_size,num_attention_heads,num_key_value_heads,rope_theta)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)
        self.input_layernorm = RMSNorm(hidden_size)
        self.post_attention_layernorm = RMSNorm(hidden_size)

    def forward(self,x):
        x = x + self.self_attn(self.input_layernorm(x))
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

#### **LlamaModel**
This code defines the `LlamaModel` class, which serves as the complete Llama-like language model. It integrates all the previously defined components (`RMSNorm`, `LlamaDecoderLayer`, and `nn.Embedding`) to form a functional sequence-to-sequence model capable of generating text.

##### `__init__` Method

*   **`vocab_size`**: The size of the vocabulary, determining the number of unique tokens the model can handle.
*   **`hidden_size`**: The dimensionality of the token embeddings and the hidden states throughout the model.
*   **`num_hidden_layers`**: The number of stacked `LlamaDecoderLayer` instances, defining the depth of the model.
*   **`tie_word_embeddings`**: A boolean flag indicating whether the input token embeddings (`embed_tokens`) should share their weights with the output projection layer (`lm_head`). Tying weights is a common practice that reduces parameters and often improves performance.
*   **`intermediate_size`**, **`num_attention_heads`**, **`num_key_value_heads`**, **`rope_theta`**: These parameters are passed down to initialize the `LlamaDecoderLayer` instances.
*   **`self.embed_tokens = nn.Embedding(vocab_size, hidden_size)`**: This layer converts input token IDs into dense vector representations (embeddings).
*   **`self.layers = nn.ModuleList([...])`**: A list of `LlamaDecoderLayer` instances, each representing one transformer block.
*   **`self.norm = RMSNorm(hidden_size)`**: A final RMS normalization layer applied to the output of the stack of decoder layers.
*   **`self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)`**: The language model head, a linear layer that projects the final hidden state back to the vocabulary size to produce logits for each token.
*   **`if tie_word_embeddings:`**: If enabled, the `lm_head`'s weights are set to be the same as `embed_tokens`'s weights.

##### `forward` Method

*   **`x = self.embed_tokens(x)`**: The input token IDs `x` are first converted into embeddings.
*   **`for layer in self.layers:`**: The embeddings are then passed sequentially through each `LlamaDecoderLayer`.
*   **`x = self.norm(x)`**: After all decoder layers, the output is normalized by the final `RMSNorm` layer.
*   **`x = self.lm_head(x)`**: Finally, the normalized output is passed through the language model head to produce the final logits, which represent the model's prediction for the next token in the sequence.

In [9]:
class LlamaModel(nn.Module):
    def __init__(self,vocab_size = 49152,hidden_size=576,num_hidden_layers = 30,tie_word_embeddings = True,intermediate_size=1536,num_attention_heads=9,num_key_value_heads=3,rope_theta=100000):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size,hidden_size)
        self.layers = nn.ModuleList([LlamaDecoderLayer(hidden_size,intermediate_size,num_attention_heads,num_key_value_heads,rope_theta) for _ in range(num_hidden_layers)])
        self.norm = RMSNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size,vocab_size,bias=False)
        if tie_word_embeddings:
            self.lm_head.weight =  self.embed_tokens.weight

    def forward(self,x):
        x = self.embed_tokens(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = self.lm_head(x)
        return x

### **Loading pretrained weights into Architecture and Checking**

This code block performs two main actions:

1.  **Model Initialization and Weight Loading**: It first creates an instance of your custom `LlamaModel`. Then, it loads the state dictionary (i.e., the learned weights and biases) from the `hf_model` (which is the pre-trained `SmolLM2-135M` model from Hugging Face) into your newly created `LlamaModel`. The `strict=False` argument is used in `load_state_dict` to allow for partial loading, which can be useful if your custom model's architecture differs slightly but shares many common layers.

2.  **Output Verification**: After loading the weights, it generates a random tensor `input_ids` to simulate a short input sequence. It then performs a forward pass with both the original `hf_model` and your `model` using these `input_ids` to obtain their respective logits (raw prediction scores). Finally, it calculates and prints the maximum absolute difference between the logits generated by the two models. A very small difference (as seen in the output) indicates that the weights have been successfully transferred and your custom `LlamaModel` produces nearly identical outputs to the reference `hf_model` for the given input.

In [10]:
model = LlamaModel()
model.load_state_dict(hf_model.model.state_dict(), strict=False)

# Test with short sequence
input_ids = torch.randint(0, 1000, (1, 5))
with torch.no_grad():
    hf_logits = hf_model(input_ids).logits
    your_logits = model(input_ids)

print("Max difference:", (hf_logits - your_logits).abs().max().item())

Max difference: 0.003612518310546875


This code block extends the previous verification step by testing the model with a longer input sequence. It generates `input_ids` of 50 tokens (compared to 5 in the previous test) and performs the same comparison.

The purpose is to confirm that after loading the weights into your custom `LlamaModel`, it continues to produce outputs that are nearly identical to the original `hf_model` even for longer sequences. A small maximum absolute difference (as shown in the output) reassures that the weight transfer and your model's forward pass logic are consistent across different input lengths.

In [11]:
input_ids = torch.randint(0, 1000, (1, 50))
with torch.no_grad():
    hf_logits = hf_model(input_ids).logits
    your_logits = model(input_ids)

print("Max difference (50 tokens):", (hf_logits - your_logits).abs().max().item())

Max difference (50 tokens): 0.010829925537109375


### Additional Training Steps

loads a pre-trained tokenizer for SmolLM2-135M and then fetches a small subset of the TinyStories dataset.

In [12]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load tokenizer for SmolLM2
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

# Load a small subset of TinyStories
dataset = load_dataset("roneneldan/TinyStories", split="train[:5000]")

print(f"Dataset size: {len(dataset)}")
print(f"Example story: {dataset[0]['text'][:200]}...")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Dataset size: 5000
Example story: One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on...


This code block is crucial for preparing our text data for model training. It tokenizes the dataset and then organizes it into mini-batches using a `DataLoader`. Let's break it down:

1.  **`from torch.utils.data import DataLoader`**: This line imports the `DataLoader` class from PyTorch, which is a utility for iterating over datasets in batches, shuffling data, and handling multi-process data loading.

2.  **`tokenize_function(examples)`**: This function defines how each example (story) in the dataset will be processed.
    *   `tokenizer.pad_token = tokenizer.eos_token`: It sets the padding token for the tokenizer to be the end-of-sequence token. This is a common practice in language modeling to ensure that shorter sequences are padded to `max_length` using a token that the model understands as the end of meaningful content.
    *   `return tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length')`: This calls the loaded tokenizer (from the previous cell) on the 'text' field of each example.
        *   `truncation=True`: Ensures that sequences longer than `max_length` are cut off.
        *   `max_length=128`: Specifies that all sequences will be truncated or padded to a length of 128 tokens.
        *   `padding='max_length'`: Instructs the tokenizer to pad shorter sequences to `max_length`.

3.  **`tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])`**: This line applies the `tokenize_function` to your `dataset`.
    *   `dataset.map(...)`: The `map` method from the `datasets` library efficiently applies a function to all elements of the dataset.
    *   `batched=True`: Means that `tokenize_function` receives multiple examples at once, which is generally more efficient for tokenization.
    *   `remove_columns=['text']`: Removes the original 'text' column from the dataset, as we now have the tokenized `input_ids` and `attention_mask`.

4.  **`tokenized_dataset.set_format('torch')`**: This sets the format of the `tokenized_dataset` to PyTorch tensors, making it directly compatible with PyTorch models.

5.  **`train_dataloader = DataLoader(tokenized_dataset, batch_size=4, shuffle=True)`**: This creates the actual `DataLoader` instance.
    *   It wraps the `tokenized_dataset`.
    *   `batch_size=4`: Specifies that the data will be loaded in batches of 4 examples.
    *   `shuffle=True`: Randomly shuffles the data at the beginning of each epoch, which is important for robust training.

6.  **`print(f"Number of batches: {len(train_dataloader)}")`**: This simply prints the total number of batches that will be processed per epoch, calculated based on the dataset size and the batch size.

In [13]:
from torch.utils.data import DataLoader

# Tokenize the dataset
def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length')

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_dataset.set_format('torch')

# Create dataloader
train_dataloader = DataLoader(tokenized_dataset, batch_size=4, shuffle=True)

print(f"Number of batches: {len(train_dataloader)}")


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Number of batches: 1250


In [14]:
# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")

Using device: cuda


In [15]:
def generate_text(prompt_text, max_length=50):
    model.eval()
    input_ids = tokenizer(prompt_text, return_tensors='pt')['input_ids'].to(device)

    with torch.no_grad():
        for _ in range(max_length):
            logits = model(input_ids)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

print(generate_text("Once upon a time"))

Once upon a time, there was a little girl named Lily. She lived in a big house with her family, but she didn't have many toys to play with. One day, her mom told her that she could go to a special place called a "school"


### Setting Optimizer

In [16]:
from torch.optim import AdamW

# Set up optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")


Using device: cuda


### Baseline training loop

### %xterm watch nvidia-smi - for GPU monitoring
**%xterm watch nvidia-smi** opens an interactive terminal within your Colab output that continuously updates and displays the current status of your NVIDIA GPU every 2 seconds. This is excellent for keeping an eye on your GPU memory and utilization during training!

In [ ]:
!pip install colab-xterm
%load_ext colabxterm
%xterm watch nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 6.4 MB/s eta 0:00:00


Launching Xterm...

<IPython.core.display.Javascript object>

In [ ]:
import time
from tqdm import tqdm

model.train()
step = 0

train_dataloader = DataLoader(tokenized_dataset, batch_size=8, shuffle=True, num_workers=2)

t_start = time.time()
pbar = tqdm(total=5000)

while step < 5000:
    for batch in train_dataloader:
        t0 = time.time()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids)
        loss = F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),
                               input_ids[:,1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        t1 = time.time()
        dt = (t1 - t0) * 1000
        tokens_per_sec = (input_ids.shape[0] * input_ids.shape[1]) / (t1 - t0)
        step += 1
        pbar.update(1)

        if step % 100 == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'tok/sec': f'{tokens_per_sec:.2f}'})

        if step >= 5000:
            break

        if step % 500 == 0:
            model.eval()
            print(generate_text("Once upon a time"))
            model.train()

pbar.close()
t_end = time.time()
total_time = t_end - t_start
total_tokens = 5000 * 4 * 128
print(f"\nTotal training time: {total_time:.2f}s")
print(f"Average tokens/sec: {total_tokens / total_time:.2f}")

torch.save({'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'step': step}, 'model_checkpoint.pt')


 10%|█         | 500/5000 [03:03<26:55,  2.79it/s, loss=1.2459, tok/sec=2880.45]

Once upon a time, there was a little girl named Lily. She loved to play with her toys and make them into pretty patterns. One day, Lily's mom gave her a big box of colorful blocks. Lily was so excited to play with them.

Lily


 20%|██        | 1000/5000 [06:04<23:54,  2.79it/s, loss=1.2191, tok/sec=2852.07]

Once upon a time, there was a little girl named Lily. She loved to play outside with her friends. One day, they went to the park to play. They saw a big slide and wanted to try it. 

Lily's friend said, "I don


 30%|███       | 1500/5000 [09:05<20:54,  2.79it/s, loss=1.0761, tok/sec=2900.60]

Once upon a time, there was a little girl named Lily. She loved to play outside in the park with her friends. One day, they found a big, shiny rock in the park. Lily picked it up and showed it to her friends.

"Look


 40%|████      | 2000/5000 [12:07<18:01,  2.78it/s, loss=0.7429, tok/sec=2860.84]

Once upon a time, there was a little girl named Lily. She loved to play with her toys and her favorite was a teddy bear. One day, she went to the park with her mom.

At the park, Lily saw a boy playing with a ball


 44%|████▍     | 2218/5000 [13:26<16:40,  2.78it/s, loss=0.9235, tok/sec=2848.20]

KeyboardInterrupt: 

### **First Optimization**: using `torch.set_float32_matmul_precision('high')`


The `torch.set_float32_matmul_precision('high')` function is a performance optimization setting in PyTorch, particularly relevant when training on modern NVIDIA GPUs (Ampere architecture and newer). Here's what it means:

*   **`float32` Matmul Precision**: This function specifically controls the precision used for `float32` (single-precision floating-point) matrix multiplication operations, which are very common in deep learning models.

*   **`'high'` Setting**: When set to `'high'`, it enables the use of **TensorFloat-32 (TF32)** arithmetic for `float32` matrix multiplications and convolutions on compatible GPUs.

    *   **What is TF32?** TF32 is a mathematical mode introduced with NVIDIA's Ampere GPUs. It combines the `float32` range (which is good for avoiding overflow/underflow) with the internal precision of `float16` (reducing the number of bits used for calculations). Essentially, it uses a 19-bit internal format (8 exponent bits like `float32`, 10 mantissa bits like `float16`, plus a sign bit).

*   **Benefits**: By using TF32, computations can be significantly faster (up to 3x faster than full `float32`) because the GPU's Tensor Cores can process these operations more efficiently. This often comes with minimal or no loss in model accuracy for most deep learning workloads, as the reduced precision usually doesn't impact convergence.

*   **Trade-off**: The primary trade-off is a slight reduction in numerical precision compared to a full `float32` operation. However, for deep learning, the performance gains typically outweigh this minor precision loss.

In [1]:
import time
from tqdm import tqdm

# this is same as `torch.set_float32_matmul_precision('high')` but
# that is deprecated, below are the new way to do it.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

model.train()
step = 0

train_dataloader = DataLoader(tokenized_dataset, batch_size=8, shuffle=True, num_workers=2)

t_start = time.time()
pbar = tqdm(total=5000)

while step < 5000:
    for batch in train_dataloader:
        t0 = time.time()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids)
        loss = F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),
                               input_ids[:,1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        t1 = time.time()
        dt = (t1 - t0) * 1000
        tokens_per_sec = (input_ids.shape[0] * input_ids.shape[1]) / (t1 - t0)
        step += 1
        pbar.update(1)

        if step % 100 == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'tok/sec': f'{tokens_per_sec:.2f}'})

        if step >= 5000:
            break

        if step % 500 == 0:
            model.eval()
            print(generate_text("Once upon a time"))
            print("\n\n")
            model.train()

pbar.close()
t_end = time.time()
total_time = t_end - t_start
total_tokens = 5000 * 4 * 128
print(f"\nTotal training time: {total_time:.2f}s")
print(f"\nAverage tokens/sec: {total_tokens / total_time:.2f}")

torch.save({'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'step': step}, 'model_checkpoint.pt')


NameError: name 'torch' is not defined

### **Second Optimization**: Autocast (Automatic Mixed Precision (AMP) training)

The 'Autocast' section refers to the use of `torch.amp.autocast` for Automatic Mixed Precision (AMP) training. This is a significant optimization technique that can dramatically speed up training and reduce memory usage on compatible hardware (like NVIDIA GPUs). Let's break down what it does:

*   **Automatic Mixed Precision (AMP)**: AMP involves performing operations in different precisions (e.g., `float16` for some operations and `float32` for others) during training. Typically, `float16` (half-precision) offers faster computations and less memory consumption, but it has a smaller dynamic range, which can lead to numerical instability issues (like underflow/overflow) for certain critical operations.

*   **`torch.amp.autocast('cuda')`**: When you wrap a block of code (like your forward pass and loss calculation) within `with autocast('cuda'):`, PyTorch automatically determines the optimal data type for each operation within that block. For operations that benefit from `float16` (like matrix multiplications), it will automatically cast inputs to `float16`. For operations that require `float32` for numerical stability (like certain accumulation steps or operations with very small gradients), it keeps them in `float32`.

*   **`GradScaler`**: AMP is often used in conjunction with `GradScaler`. Because `float16` has a limited range, gradients can become extremely small and underflow to zero, leading to a stalled training process. `GradScaler` addresses this by:
    1.  **Scaling the loss**: It multiplies the loss by a large factor before calling `loss.backward()`. This makes the gradients larger, preventing them from underflowing when represented in `float16`.
    2.  **Unscaling the gradients**: Before the optimizer updates the model's weights, `GradScaler` divides the gradients by the same scaling factor. This ensures that the weight updates are correct.
    3.  **Dynamic scaling**: `GradScaler` can also dynamically adjust the scaling factor during training, increasing it when gradients are healthy and decreasing it if it detects `NaN` or `inf` gradients (which might indicate an overflow).

**Benefits of using Autocast and GradScaler:**

*   **Faster Training**: By utilizing `float16` where appropriate, operations can be processed more quickly on Tensor Cores (if available on your GPU).
*   **Reduced Memory Usage**: `float16` uses half the memory of `float32`, allowing you to train with larger batch sizes or larger models.
*   **Improved Numerical Stability**: `GradScaler` mitigates the numerical issues that could arise from using `float16` alone.

In summary, Autocast and GradScaler provide a powerful and convenient way to enable mixed-precision training, leading to significant performance gains while maintaining training stability.

In [ ]:
from torch.amp import autocast, GradScaler

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Initialize gradient scaler
scaler = GradScaler()

print("Before training:")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

model.train()
step = 0
t_start = time.time()
pbar = tqdm(total=5000)

while step < 5000:
    for batch in train_dataloader:
        t0 = time.time()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Wrap forward pass in autocast
        with autocast('cuda'):
            logits = model(input_ids)
            loss = F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),
                                   input_ids[:,1:].reshape(-1))

        optimizer.zero_grad()
        scaler.scale(loss).backward()  # Scale loss for fp16
        scaler.step(optimizer)         # Unscale and step
        scaler.update()                # Update scaler

        t1 = time.time()
        dt = (t1 - t0) * 1000
        tokens_per_sec = (input_ids.shape[0] * input_ids.shape[1]) / (t1 - t0)
        step += 1
        pbar.update(1)

        if step % 100 == 0:
            gpu_mem = torch.cuda.memory_allocated() / 1e9
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'tok/sec': f'{tokens_per_sec:.2f}', 'GPU': f'{gpu_mem:.2f}GB\n'})

        if step >= 5000:
            break

        if step % 500 == 0:
            model.eval()
            print(generate_text("Once upon a time"))
            model.train()

pbar.close()
t_end = time.time()
total_time = t_end - t_start
total_tokens = 5000 * 4 * 128
print(f"\nTotal training time: {total_time:.2f}s")
print(f"\nAverage tokens/sec: {total_tokens / total_time:.2f}")
print(f"\nFinal GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Before training:
GPU memory allocated: 2.33 GB
GPU memory reserved: 5.39 GB



  9%|▊         | 437/5000 [02:01<21:13,  3.58it/s, loss=0.1157, tok/sec=6007.97, GPU=2.32GB]

 10%|█         | 501/5000 [01:36<49:29,  1.51it/s, loss=0.1313, tok/sec=6048.60, GPU=2.32GB]

Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she went to the park with her mom. She saw a bird's nest in a tree and asked her mom, "What is that?"




 20%|██        | 1001/5000 [03:15<44:33,  1.50it/s, loss=0.1007, tok/sec=6149.72, GPU=2.32GB]

Once upon a time, there was a little girl named Lily. She loved to play with her toys and her friends. One day, Lily and her friends were playing in the park when they saw a big, scary dog. 

Lily's friend, Tommy, said


...

Once upon a time, there was a little girl named Lily. She loved to play with her toys and run around outside. One day, she went to the park with her mom.

At the park, Lily met a little boy named Tim. Tim was very



 90%|█████████ | 4501/5000 [14:10<05:32,  1.50it/s, loss=0.0817, tok/sec=6015.44, GPU=2.32GB]

Once upon a time, there was a little girl named Lily. She loved to play outside in the rain. One day, her mom asked her to come inside because it was wet outside. 

Lily said, "But mommy, I love the rain! Can't



100%|██████████| 5000/5000 [15:43<00:00,  5.30it/s, loss=0.1136, tok/sec=6071.70, GPU=2.32GB]


Total training time: 943.02s

Average tokens/sec: 2714.69

Final GPU memory: 2.32 GB


### **Third Optimization**: torch.compile

`torch.compile` is a powerful new feature introduced in PyTorch 2.0 that aims to significantly speed up your PyTorch code without requiring changes to your model. Here's a breakdown of what it does:

*   **What is it?** `torch.compile` is a Python API that takes a PyTorch module or function and returns a compiled version of it. When you call this compiled version, it executes much faster.

*   **How it Works (JIT Compilation)**: Under the hood, `torch.compile` uses Just-In-Time (JIT) compilation techniques. When you pass your model to `torch.compile`, it first traces the execution of your model's forward (and sometimes backward) pass. During this tracing, it identifies the sequence of PyTorch operations.

*   **Graph Capture and Optimization**: Instead of running these operations one by one as interpreted Python code, `torch.compile` captures them into a computational graph. This graph is then passed to a backend (like TorchInductor, which is the default for CUDA). This backend optimizes the graph by:
    *   **Kernel Fusion**: Combining multiple small operations into a single, larger GPU kernel call, reducing overhead.
    *   **Memory Optimization**: Arranging memory accesses more efficiently.
    *   **Code Generation**: Generating highly optimized C++ or CUDA code for the fused kernels, tailored to the specific GPU architecture.

*   **Dynamic Shapes**: `torch.compile` is designed to handle dynamic input shapes (e.g., variable batch sizes or sequence lengths), recompiling efficiently if new shapes are encountered.

*   **Benefits**:
    *   **Significant Speedups**: Often provides 1.5x to 2x (or even more) speedups for training and inference, especially on GPUs.
    *   **Ease of Use**: It's typically a one-line change (`model = torch.compile(model)`).
    *   **Compatibility**: Works with existing PyTorch code and popular libraries.
    *   **Reduced Overhead**: Minimizes the Python interpreter overhead by offloading more work to optimized kernels.

In essence, `torch.compile` acts as an intelligent optimizer that turns your high-level PyTorch code into highly efficient low-level GPU operations, leading to faster execution times.

In [17]:
# Compile the model (add this after moving model to device, before training)
model = torch.compile(model)


from torch.amp import autocast, GradScaler

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Initialize gradient scaler
scaler = GradScaler()

print("Before training:")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

model.train()
step = 0
t_start = time.time()
pbar = tqdm(total=5000)

while step < 5000:
    for batch in train_dataloader:
        t0 = time.time()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Wrap forward pass in autocast
        with autocast('cuda'):
            logits = model(input_ids)
            loss = F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),
                                   input_ids[:,1:].reshape(-1))

        optimizer.zero_grad()
        scaler.scale(loss).backward()  # Scale loss for fp16
        scaler.step(optimizer)         # Unscale and step
        scaler.update()                # Update scaler

        t1 = time.time()
        dt = (t1 - t0) * 1000
        tokens_per_sec = (input_ids.shape[0] * input_ids.shape[1]) / (t1 - t0)
        step += 1
        pbar.update(1)

        if step % 100 == 0:
            gpu_mem = torch.cuda.memory_allocated() / 1e9
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'tok/sec': f'{tokens_per_sec:.2f}', 'GPU': f'{gpu_mem:.2f}GB\n'})

        if step >= 5000:
            break

        if step % 500 == 0:
            model.eval()
            print(generate_text("Once upon a time"))
            model.train()

pbar.close()
t_end = time.time()
total_time = t_end - t_start
total_tokens = 5000 * 4 * 128
print(f"\nTotal training time: {total_time:.2f}s")
print(f"\nAverage tokens/sec: {total_tokens / total_time:.2f}")
print(f"\nFinal GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")



/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


Before training:
GPU memory allocated: 0.55 GB
GPU memory reserved: 0.61 GB


 10%|█         | 501/5000 [03:21<12:09:33,  9.73s/it, loss=1.5801, tok/sec=5310.98, GPU=2.29GB]

Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she went outside to play with her friends. They all had a big game of tag. Lily was the fastest one. She ran around and around


 20%|██        | 1002/5000 [04:14<13:19,  5.00it/s, loss=1.7254, tok/sec=5244.75, GPU=2.29GB]

...

Once upon a time, there was a little girl named Lily. She loved to play outside in the warm sun. One day, she went to the park with her mom. They saw a big tree with lots of apples on it. Lily wanted to pick one of the


 90%|█████████ | 4501/5000 [10:23<02:03,  4.05it/s, loss=0.9454, tok/sec=5156.68, GPU=2.29GB]

Once upon a time, there was a little girl named Lily. She loved to play outside in the park with her friends. One day, Lily saw a big, red ball. She wanted to catch it, but she didn't have a net to help her.



100%|██████████| 5000/5000 [11:15<00:00,  7.40it/s, loss=1.0602, tok/sec=5176.32, GPU=2.28GB]


Total training time: 675.74s

Average tokens/sec: 3788.46

Final GPU memory: 2.28 GB


### Saving model

In [18]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'step': step
}, 'checkpoint.pt')


### Loading a Checkpoint:

In [19]:
checkpoint = torch.load('checkpoint.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
step = checkpoint['step']

### Training for 50 more steps

In [20]:
model.train()
target_steps = step + 50

while step < target_steps:
    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids)
        loss = F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),
                               input_ids[:,1:].reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        step += 1

        if step >= target_steps:
            break

        if step % 10 == 0:  # Print more frequently since it's only 50 steps
            print(f"Step {step}, Loss: {loss.item():.4f}")

print(f"\nFinal step: {step}")
model.eval()
print("\n")
print(generate_text("Once upon a time"))


Step 5010, Loss: 0.9968
Step 5020, Loss: 0.7218
Step 5030, Loss: 0.8190
Step 5040, Loss: 0.8055

Final step: 5050


Once upon a time, there was a little girl named Lily. She loved to play with her toys and eat cookies. One day, she saw a big, red button on the wall. She thought it was very cool and wanted to touch it.

Lily pushed


### **Save float16 for inference**

In [ ]:
import torch
import os

# 1. Access the 'model_state_dict'
model_state_dict = model.state_dict()

# 2. Create an empty dictionary for float16 state dict
float16_state_dict = {}

# 3. iterate, clean keys, and convert to float16
for k, v in model_state_dict.items():
    clean_k = k.replace('_orig_mod.', '') if '_orig_mod.' in k else k
    # 6. Convert the tensor to torch.float16 precision
    float16_state_dict[clean_k] = v.to(torch.float16)

# 4. Save the new float16 state dictionary
torch.save(float16_state_dict, "llama_model_weights.pt")
# print("LlamaModel weights saved to: llama_model_weights.pt with float16 precision.")



### Running with **Pytorch profiler**

The PyTorch Profiler, particularly with `torch.profiler.profile`, `record_function`, and `ProfilerActivity`, is a powerful tool for analyzing the performance of your PyTorch code. It helps you understand where time is being spent and identify bottlenecks. Let's break down these elements:

*   **`torch.profiler.profile`**: This is a context manager that enables and disables the profiler. You wrap the code you want to profile within a `with profile(...) as prof:` block. Key parameters include:
    *   `activities`: A list of `ProfilerActivity` enums indicating what to collect data for (e.g., `ProfilerActivity.CPU` for CPU operations, `ProfilerActivity.CUDA` for GPU operations).
    *   `record_shapes`: (Optional) If `True`, the profiler records input shapes to operators. Useful for understanding how different input sizes affect performance.
    *   `profile_memory`: (Optional) If `True`, the profiler tracks memory allocations and deallocations.

*   **`torch.profiler.record_function`**: This is another context manager used to add custom labels or ranges to the profiling timeline. By wrapping specific sections of your code (like 'data_loading', 'forward', 'backward' in the example), you can easily see how much time and resources are consumed by each logical step of your program.

*   **`torch.profiler.ProfilerActivity`**: This is an Enum that allows you to specify which types of operations the profiler should record. Common values are `CPU` (for Python, C++ operations), `CUDA` (for GPU kernels and memory transfers), and `VULKAN` (for Vulkan backend, if used).

**How they work together (as seen in the code):**

1.  The `with profile(...) as prof:` block starts the profiling session. It's configured to record both CPU and CUDA activities.
2.  Inside the training loop, `with record_function("data_loading")`, `with record_function("forward")`, and `with record_function("backward")` explicitly mark different phases of each training step. This allows the profiler to attribute time and resources accurately to these custom-named blocks.
3.  After the profiling session (when the `with profile` block exits), the `prof.key_averages().table(...)` method is used to process and print the collected profiling data in a human-readable table format, sorted by specified metrics (e.g., `cuda_time_total` or `cpu_time_total`). This helps pinpoint performance bottlenecks.

In [ ]:
import time
from tqdm import tqdm
from torch.amp import autocast, GradScaler
from torch.profiler import profile, record_function, ProfilerActivity
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# reduced batch size as profiler takes additional memory
train_dataloader = DataLoader(tokenized_dataset, batch_size=1, shuffle=True, num_workers=2)

print(type(model))
model = model.to(device)
# disable compile while running with profiler
# # Compile the model (add this after moving model to device, before training)
# model = torch.compile(model)

# need to Recreate optimizer before profiling
optimizer = AdamW(model.parameters(), lr=5e-5)

scaler = GradScaler(enabled=True)

print("Before training:")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model.train()
step = 0
t_start = time.time()
pbar = tqdm(total=100)  # Profile just 100 steps

# Start profiler
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
             record_shapes=False,
             profile_memory=False) as prof:

    while step < 20:
        for batch in train_dataloader:
            with record_function("data_loading"):
                t0 = time.time()
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

            with record_function("forward"):
                with autocast('cuda'):
                    logits = model(input_ids)
                    loss = F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),
                                           input_ids[:,1:].reshape(-1))

            with record_function("backward"):
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            t1 = time.time()
            tokens_per_sec = (input_ids.shape[0] * input_ids.shape[1]) / (t1 - t0)
            step += 1
            pbar.update(1)

            if step >= 100:
                break

pbar.close()
t_end = time.time()
print(f"\nProfiling complete. Total time: {t_end - t_start:.2f}s")

# Print profiling results
print("\n=== Top operations by CUDA time ===")
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))

print("\n=== Top operations by CPU time ===")
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


<class '__main__.LlamaModel'>
Before training:
GPU memory allocated: 0.55 GB


100%|██████████| 100/100 [00:57<00:00,  1.73it/s]



Profiling complete. Total time: 57.90s

=== Top operations by CUDA time ===


In [ ]:
pip install pytorch-lightning tensorboard


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 70.5 MB/s eta 0:00:00


### Converting to **PyTorch Lightning**

PyTorch Lightning is a lightweight PyTorch wrapper that provides a high-level interface for training complex neural networks. It abstracts away boilerplate code, allowing researchers and engineers to focus more on model design and less on the intricate details of the training loop.

**Why convert to PyTorch Lightning?**

*   **Boilerplate Reduction**: It significantly reduces the amount of repetitive code for training, validation, testing, and logging.
*   **Reproducibility**: Encourages structured code that is easier to reproduce and share.
*   **Scalability**: Simplifies distributed training across multiple GPUs or TPUs with minimal code changes.
*   **Best Practices**: Enforces good practices like using `torch.no_grad()`, `model.eval()`, and proper gradient accumulation.
*   **Advanced Features**: Provides built-in support for features like Automatic Mixed Precision (AMP), gradient clipping, logging (TensorBoard, Weights & Biases), and checkpointing.

**Key Components involved in the conversion:**

1.  **`pl.LightningModule`**: This class encapsulates the model (`nn.Module`), the forward pass, and all aspects of the optimization loop (training step, validation step, test step, and optimizer configuration).
2.  **`pl.LightningDataModule`**: This class handles all data-related aspects, including data loading, preprocessing, and creating `DataLoader` instances for training, validation, and testing. It ensures data is prepared consistently.
3.  **`pl.Trainer`**: This orchestrates the entire training process. You pass the `LightningModule`, `LightningDataModule`, and various callbacks (like checkpointing or progress bars) to the `Trainer`, which then manages the execution of the training loop.

In [ ]:
import pytorch_lightning as pl
from torch.optim import AdamW

class LlamaLightningModule(pl.LightningModule):
    def __init__(self, model, learning_rate=5e-5):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        # Forward pass
        logits = self.model(input_ids)

        # Calculate loss
        loss = F.cross_entropy(
            logits[:, :-1, :].reshape(-1, logits.shape[-1]),
            input_ids[:, 1:].reshape(-1)
        )

        # Log the loss
        self.log('train_loss', loss, prog_bar=True)

        return loss

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.learning_rate)
        return optimizer


In [ ]:
class TinyStoriesDataModule(pl.LightningDataModule):
    def __init__(self, tokenizer, batch_size=8, num_workers=2):
        super().__init__()
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.num_workers = num_workers

    # Tokenize the dataset
    def tokenize_function(self, examples):
        self.tokenizer.pad_token = self.tokenizer.eos_token
        return self.tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length')

    def setup(self, stage=None):
        # Load and tokenize dataset
        # dataset = load_dataset("roneneldan/TinyStories", split="train[:5000]")
        self.train_dataset = dataset.map(self.tokenize_function, batched=True, remove_columns=['text'])
        self.train_dataset.set_format('torch') # donot return set as instance variable

    def train_dataloader(self):
        # Return the training dataloader
        # Create dataloader
        train_dataloader = DataLoader(self.train_dataset, batch_size=self.batch_size,num_workers=self.num_workers, shuffle=True)
        return train_dataloader



In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, RichProgressBar
from pytorch_lightning.loggers import TensorBoardLogger

# Create logger
logger = TensorBoardLogger("tb_logs", name="smollm_training")

# Create checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/",
    filename="smollm-{step}",
    every_n_train_steps=500,
    save_top_k=-1  # Save all checkpoints, or use save_top_k=1 with monitor='train_loss'
)

# Create trainer
trainer = pl.Trainer(
    max_steps=5000,
    logger=logger,
    callbacks=[checkpoint_callback, RichProgressBar()],
    precision="16-mixed",  # For autocast
    log_every_n_steps=100
)


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


In [ ]:
# Initialize your module and data
lightning_model = LlamaLightningModule(model, learning_rate=5e-5)
data_module = TinyStoriesDataModule(tokenizer, batch_size=8)

# Train!
trainer.fit(lightning_model, data_module)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model │ LlamaModel │  134 M │ train │
└───┴───────┴────────────┴────────┴───────┘

Trainable params: 134 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 134 M                                                                                                
Total estimated model params size (MB): 538                                                                        
Modules in train mode: 425                                                                                         
Modules in eval mode: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=5000` reached.


In [ ]:
import pytorch_lightning as pl
from torch.optim import AdamW

class LlamaLightningModule(pl.LightningModule):
    def __init__(self, model, learning_rate=5e-5):
        super().__init__()
        self.model = torch.compile(model)
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        # Forward pass
        logits = self.model(input_ids)

        # Calculate loss
        loss = F.cross_entropy(
            logits[:, :-1, :].reshape(-1, logits.shape[-1]),
            input_ids[:, 1:].reshape(-1)
        )

        # Log the loss
        self.log('train_loss', loss, prog_bar=True)

        return loss

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.learning_rate)
        return optimizer


In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, RichProgressBar
from pytorch_lightning.loggers import TensorBoardLogger

# Create logger
logger = TensorBoardLogger("tb_logs", name="smollm_training")

# Create checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/",
    filename="smollm-final",
    save_last=True,  # Save only the last checkpoint
    every_n_train_steps=None  # Remove periodic saving
)

# Create trainer
trainer = pl.Trainer(
    max_steps=5000,
    logger=logger,
    callbacks=[checkpoint_callback, RichProgressBar()],
    precision="16-mixed",  # For autocast
    log_every_n_steps=100
)


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


In [ ]:
# Initialize your module and data
lightning_model = LlamaLightningModule(model, learning_rate=5e-5)
data_module = TinyStoriesDataModule(tokenizer, batch_size=8)

# Train!
trainer.fit(lightning_model, data_module)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /content/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type            ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model │ OptimizedModule │  134 M │ train │
└───┴───────┴─────────────────┴────────┴───────┘

Trainable params: 134 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 134 M                                                                                                
Total estimated model params size (MB): 538                                                                        
Modules in train mode: 426                                                                                         
Modules in eval mode: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=5000` reached.


In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, RichProgressBar
from pytorch_lightning.loggers import TensorBoardLogger

# Create logger
logger = TensorBoardLogger("tb_logs", name="smollm_training")

# Create checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/",
    filename="smollm-final",
    save_last=True,  # Save only the last checkpoint
    every_n_train_steps=None  # Remove periodic saving
)

# Create trainer
trainer = pl.Trainer(
    max_steps=5000,
    logger=logger,
    callbacks=[checkpoint_callback, RichProgressBar()],
    precision="16-mixed",  # For autocast
    log_every_n_steps=100
)


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Initialize your module and data
lightning_model = LlamaLightningModule(model, learning_rate=5e-5)
data_module = TinyStoriesDataModule(tokenizer, batch_size=8)

# Train!
trainer.fit(lightning_model, data_module)

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /content/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type            ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model │ OptimizedModule │  134 M │ train │
└───┴───────┴─────────────────┴────────┴───────┘

Trainable params: 134 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 134 M                                                                                                
Total estimated model params size (MB): 538                                                                        
Modules in train mode: 426                                                                                         
Modules in eval mode: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=5000` reached.


**Reasoning**:
Save the trained model's state dictionary and the tokenizer to local files as instructed.



**Reasoning**:
Save the original model's state dictionary and the tokenizer to disk as instructed.



### gradio inference script - bundling model for inference

In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
import gradio as gr
import os

# --- 1. Define Llama Architecture Classes (Copy-pasted from your notebook) ---

class RMSNorm(nn.Module):
    def __init__(self, embedd_dim, eps=1e-05):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(embedd_dim))
        self.eps = eps

    def forward(self, x):
        x = x / (torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True)) + self.eps)
        x = self.weight * x
        return x

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim=64, rope_theta=100000):
        super().__init__()
        self.head_dim = head_dim
        self.rope_theta = rope_theta
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x, position_ids):
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1)
        position_ids_expanded = position_ids[:, None, :].float()

        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos()
        sin = emb.sin()

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)

def rotate_half(x):
    x1 = x[..., :x.shape[-1]//2]
    x2 = x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class LlamaAttention(nn.Module):
    def __init__(self,hidden_size,num_attention_heads=9,num_key_value_heads=3, rope_theta=100000):
        super().__init__()
        assert hidden_size%num_attention_heads == 0
        assert hidden_size%num_key_value_heads == 0
        self.hidden_size = hidden_size
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.head_dim = self.hidden_size//self.num_attention_heads
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, self.num_key_value_heads*self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_size, self.num_key_value_heads*self.head_dim, bias=False)
        self.o_proj = nn.Linear(hidden_size,hidden_size, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, rope_theta)

    def forward(self, x):
        batch, seq_len, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch, seq_len, self.num_attention_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch, -1)
        cos, sin = self.rope(q, position_ids)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        k = k.repeat_interleave(self.num_attention_heads // self.num_key_value_heads, dim=1)
        v = v.repeat_interleave(self.num_attention_heads // self.num_key_value_heads, dim=1)

        attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch, seq_len, -1)
        attn_output = self.o_proj(attn_output)

        return attn_output

class LlamaMLP(nn.Module):
    def __init__(self,hidden_size=576,intermediate_size=1536):
        super().__init__()
        self.gate_proj =  nn.Linear(hidden_size,intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size,intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size,bias=False)
        self.silu = nn.SiLU()

    def forward(self,x):
        x_1 = self.silu(self.gate_proj(x))
        x_2 = self.up_proj(x)
        return self.down_proj(x_1*x_2)

class LlamaDecoderLayer(nn.Module):
    def __init__(self, hidden_size=576,intermediate_size=1536,num_attention_heads=9,num_key_value_heads=3, rope_theta=100000):
        super().__init__()
        self.self_attn = LlamaAttention(hidden_size,num_attention_heads,num_key_value_heads,rope_theta)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)
        self.input_layernorm = RMSNorm(hidden_size)
        self.post_attention_layernorm = RMSNorm(hidden_size)

    def forward(self,x):
        x = x + self.self_attn(self.input_layernorm(x))
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

class LlamaModel(nn.Module):
    def __init__(self,vocab_size = 49152,hidden_size=576,num_hidden_layers = 30,tie_word_embeddings = True,intermediate_size=1536,num_attention_heads=9,num_key_value_heads=3,rope_theta=100000):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size,hidden_size)
        self.layers = nn.ModuleList([LlamaDecoderLayer(hidden_size,intermediate_size,num_attention_heads,num_key_value_heads,rope_theta) for _ in range(num_hidden_layers)])
        self.norm = RMSNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size,vocab_size,bias=False)
        if tie_word_embeddings:
            self.lm_head.weight =  self.embed_tokens.weight

    def forward(self,x):
        x = self.embed_tokens(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = self.lm_head(x)
        return x

# --- 2. Load Model and Tokenizer ---

# Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
tokenizer.pad_token = tokenizer.eos_token # Ensure pad_token is set for consistent behavior

# Initialize the model with the same architecture parameters
model = LlamaModel()

# Load the saved state dictionary
model_weights = torch.load("llama_model_weights.pt", map_location=device)
model = LlamaModel()

# # cleaning of key names in weights state dict
# cleaned_state_dict = {}
# for k, v in model_weights.items():
#     if k.startswith('_orig_mod.'):
#         cleaned_state_dict[k.replace('_orig_mod.', '')] = v
#     else:
#         cleaned_state_dict[k] = v

# model_weights = cleaned_state_dict

print("'_orig_mod.' prefix successfully removed from state_dict keys.")

model.load_state_dict(model_weights)
model.to(device)
model.eval() # Set model to evaluation mode

print(f"Model and tokenizer loaded on {device}.")

# --- 3. Define Text Generation Function ---

def generate_text(prompt_text, max_length=100, temperature=0.7, top_k=50):
    input_ids = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=128)['input_ids'].to(device)

    generated_ids = input_ids # Start with the prompt's input_ids

    with torch.no_grad():
        for _ in range(max_length - input_ids.shape[1]): # Generate up to max_length tokens
            logits = model(generated_ids)
            last_logits = logits[:, -1, :] # Logits for the last token

            # Apply temperature
            if temperature > 0:
                last_logits = last_logits / temperature

            # Apply top-k filtering
            if top_k > 0:
                values, indices = torch.topk(last_logits, top_k)
                last_logits = torch.full_like(last_logits, -float('inf'))
                last_logits.scatter_(1, indices, values)

            probabilities = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1) # Sample from distribution

            if next_token.item() == tokenizer.eos_token_id:
                break # Stop if EOS token is generated

            generated_ids = torch.cat([generated_ids, next_token], dim=1)

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# --- 4. Create Gradio Interface ---

iface = gr.Interface(
    fn=generate_text,
    inputs=[
        gr.Textbox(lines=2, placeholder="Enter your prompt here...", label="Prompt"),
        gr.Slider(minimum=10, maximum=200, value=100, step=10, label="Max Length"),
        gr.Slider(minimum=0.1, maximum=2.0, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(minimum=0, maximum=100, value=50, step=1, label="Top-K Sampling"),
    ],
    outputs="text",
    title="Fine-tuned SmolLM2-135M Text Generator",
    description="Generate text with a custom-trained SmolLM2 model. Adjust parameters for different generation styles."
)

if __name__ == "__main__":
    iface.launch(share=True)

'_orig_mod.' prefix successfully removed from state_dict keys.
Model and tokenizer loaded on cuda.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://db02f28b5438f5b65b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
